# MiniMind-O 部署与推理 (Hugging Face 版)
本笔记本用于在 Google Colab 环境中快速克隆、下载权重并部署 MiniMind-O 全模态模型。

## 步骤 1: 克隆项目并安装环境依赖

In [ ]:
# 1. 克隆官方仓库
!git clone --depth 1 https://github.com/jingyaogong/minimind-o
%cd minimind-o

# 2. 安装项目必须的依赖环境
!pip install -r requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple

# 3. 升级或安装 Hugging Face 官方下载工具
!pip install -U huggingface_hub

## 步骤 2: 从 Hugging Face 下载模型权重

In [ ]:
import os

# 如果国内连接 HF 较慢，可以取消下面这行的注释来使用镜像源：
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

# 创建项目结构所需的文件夹
os.makedirs("./model/SenseVoiceSmall", exist_ok=True)
os.makedirs("./model/siglip2-base-p32-256-ve", exist_ok=True)
os.makedirs("./out", exist_ok=True)

print("开始从 Hugging Face 下载模型组件...")

# 1. 下载 SenseVoiceSmall 语音编码器
!huggingface-cli download FunAudioLLM/SenseVoiceSmall --local-dir ./model/SenseVoiceSmall --local-dir-use-symlinks False

# 2. 下载 SigLIP2 视觉编码器
!huggingface-cli download google/siglip2-base-patch32-256 --local-dir ./model/siglip2-base-p32-256-ve --local-dir-use-symlinks False

# 3. 下载 MiniMind-O 核心模型及配套权重
!huggingface-cli download jingyaogong/minimind-3o --local-dir ./out --local-dir-use-symlinks False

print("模型权重下载完成！")

## 步骤 3: 开启 Gradio 临时公网链接并启动 WebUI 部署

In [ ]:
import re
import os

demo_file = 'web_demo.py'
if os.path.exists(demo_file):
    with open(demo_file, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # 自动定位并将 demo.launch(...) 修改为支持公网分享的 demo.launch(share=True)
    if 'launch(' in content and 'share=True' not in content:
        content = re.sub(r'\.launch\((.*?)\)', r'.launch(\1, share=True)', content)
        with open(demo_file, 'w', encoding='utf-8') as f:
            f.write(content)
        print("已成功为 WebUI 注入 Gradio 公网分享模式 (share=True)")

print("正在启动 Web 运行脚本，请等待下方输出中的 public URL 链接...")
# 运行部署脚本
!python web_demo.py